# 三重項-三重項消滅(TTA)過程の量子ダイナミクスシミュレーション

このチュートリアルでは、二つの色素分子A、Bにおける励起エネルギー移動と三重項-三重項消滅(Triplet-Triplet Annihilation, TTA)過程の量子ダイナミクスを、古典的、qubit、qudit表現で記述し、鈴木-トロッター分解を用いてシミュレーションします。

## 目次

1. [理論的背景](#1-理論的背景)
2. [古典的速度論モデル](#2-古典的速度論モデル)
3. [量子力学的記述](#3-量子力学的記述)
4. [Qubit表現によるシミュレーション（Qiskit）](#4-qubit表現によるシミュレーションqiskit)
5. [Qudit表現によるシミュレーション（MQT）](#5-qudit表現によるシミュレーションmqt)
6. [結果の比較と考察](#6-結果の比較と考察)
7. [まとめ](#7-まとめ)

詳細な理論は [`triplet_triplet_annihilation_theory.md`](triplet_triplet_annihilation_theory.md) を参照してください。


In [ ]:
# 必要なライブラリのインポート
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.linalg import expm
import sys
import os
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# QuTiPのインポート
import qutip as qt

# Qiskitのインポート（qubit表現用）
try:
    from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
    from qiskit.quantum_info import Statevector, Operator
    from qiskit.visualization import circuit_drawer
    from qiskit_aer import Aer
    QISKIT_AVAILABLE = True
    print("✓ Qiskit is available")
except ImportError as e:
    print(f"✗ Qiskit not found: {e}")
    print("  Qubit simulations will be skipped.")
    QISKIT_AVAILABLE = False

# MQT（Munich Quantum Toolkit）のインポート（qudit表現用）
try:
    sys.path.insert(0, os.path.abspath('../..'))
    from qudit.qudit.mqt_simulator import MQTSimulator
    from qudit.qudit.trotter_decomposition import SuzukiTrotterDecomposition as QuditTrotter
    from qudit.qudit.circuit_visualization import CircuitVisualizer as QuditVisualizer
    MQT_AVAILABLE = True
    print("✓ MQT modules are available")
except ImportError as e:
    print(f"✗ MQT modules not found: {e}")
    print("  Qudit simulations will be skipped.")
    MQT_AVAILABLE = False

# プロット設定
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['lines.linewidth'] = 2

print("\n環境設定完了")
print(f"NumPy version: {np.__version__}")
print(f"QuTiP version: {qt.__version__}")


## 1. 理論的背景

### 1.1 分子の電子状態

色素分子は以下の3つの電子状態を持ちます：

- **基底一重項状態 (S₀)**: 全電子がスピン対を形成し、全スピン角運動量が0
- **励起三重項状態 (T₁)**: 二つの不対電子が平行スピンを持ち、全スピン角運動量が1
- **励起一重項状態 (S₁)**: 電子が励起されているが、スピン対が保たれている

エネルギー順位: $E_{S_0} < E_{T_1} < E_{S_1}$

### 1.2 物理過程

**エネルギー移動**: 
$$|T_1 S_0\rangle \leftrightarrow |S_0 T_1\rangle$$

分子Aが励起三重項状態、分子Bが基底状態にある場合、エネルギーが分子Aから分子Bに移動し、逆の状態になります。

**三重項-三重項消滅 (TTA)**: 
$$|T_1 T_1\rangle \rightarrow |S_1 S_0\rangle + |S_0 S_1\rangle$$

両方の分子が励起三重項状態にある場合、相互作用により一方が励起一重項状態に、他方が基底状態に遷移します。

### 1.3 初期条件

初期状態：
- 分子A: 励起三重項状態 $|T_1\rangle$
- 分子B: 基底一重項状態 $|S_0\rangle$

2分子系の初期状態： $|\psi(0)\rangle = |T_1 S_0\rangle$

### 1.4 状態空間

2分子系の全状態空間は9次元（各分子が3準位）：

$$\{|S_0 S_0\rangle, |S_0 T_1\rangle, |S_0 S_1\rangle, |T_1 S_0\rangle, |T_1 T_1\rangle, |T_1 S_1\rangle, |S_1 S_0\rangle, |S_1 T_1\rangle, |S_1 S_1\rangle\}$$

本シミュレーションでは、物理的に重要な以下の状態に注目します：
- $|T_1 S_0\rangle$: 初期状態
- $|S_0 T_1\rangle$: エネルギー移動後
- $|T_1 T_1\rangle$: 両分子励起状態
- $|S_1 S_0\rangle, |S_0 S_1\rangle$: TTA生成物


In [ ]:
# パラメータ設定# エネルギーパラメータ (単位: eV)E_S0 = 0.0          # 基底一重項状態のエネルギー（基準）E_T1 = 1.5          # 励起三重項状態のエネルギーE_S1 = 2.2          # 励起一重項状態のエネルギー# 相互作用パラメータ (単位: eV)V_ET = 0.05         # エネルギー移動結合定数V_TTA = 0.03        # 三重項-三重項消滅結合定数V_form = 0.02       # T1T1形成結合定数（NEW: TTAを可能にするために追加）# 時間パラメータt_max = 100.0       # 最大時間 (単位: ps)n_steps = 200       # 時間ステップ数tlist = np.linspace(0, t_max, n_steps)# 鈴木-トロッター分解パラメータtrotter_order = 2   # トロッター分解の次数 (1, 2, or 4)dt_trotter = 0.5    # トロッターステップサイズ (ps)n_trotter_steps = int(t_max / dt_trotter)# ショットシミュレーションパラメータn_shots = 10000     # 測定ショット数print("パラメータ設定:")print(f"  エネルギー準位:")print(f"    E_S0 = {E_S0} eV")print(f"    E_T1 = {E_T1} eV")print(f"    E_S1 = {E_S1} eV")print(f"  相互作用:")print(f"    V_ET  = {V_ET} eV (エネルギー移動)")print(f"    V_TTA = {V_TTA} eV (TTA)")print(f"  時間発展:")print(f"    t_max = {t_max} ps")print(f"    Trotterステップ = {dt_trotter} ps")print(f"    Trotter次数 = {trotter_order}")print(f"    ショット数 = {n_shots}")

## 2. 古典的速度論モデル

### 2.1 速度方程式

古典的な速度論モデルでは、各状態のポピュレーションの時間発展を微分方程式で記述します。簡略化したモデルでは、主要な5つの状態を考慮します：

- $p_{T_1 S_0}(t)$: 分子Aが$T_1$、分子Bが$S_0$の確率
- $p_{S_0 T_1}(t)$: 分子Aが$S_0$、分子Bが$T_1$の確率
- $p_{T_1 T_1}(t)$: 両分子が$T_1$の確率
- $p_{S_1 S_0}(t)$: 分子Aが$S_1$、分子Bが$S_0$の確率
- $p_{S_0 S_1}(t)$: 分子Aが$S_0$、分子Bが$S_1$の確率

速度方程式：

$$\frac{dp_{T_1 S_0}}{dt} = -k_{\text{ET}} p_{T_1 S_0} + k_{\text{ET}} p_{S_0 T_1}$$

$$\frac{dp_{S_0 T_1}}{dt} = k_{\text{ET}} p_{T_1 S_0} - k_{\text{ET}} p_{S_0 T_1}$$

$$\frac{dp_{T_1 T_1}}{dt} = k_{\text{form}} p_{T_1 S_0} p_{S_0 T_1} - 2k_{\text{TTA}} p_{T_1 T_1}$$

$$\frac{dp_{S_1 S_0}}{dt} = k_{\text{TTA}} p_{T_1 T_1}$$

$$\frac{dp_{S_0 S_1}}{dt} = k_{\text{TTA}} p_{T_1 T_1}$$

ここで、$k_{\text{ET}}$はエネルギー移動速度定数、$k_{\text{TTA}}$はTTA速度定数、$k_{\text{form}}$は$T_1T_1$状態形成速度定数です。


In [ ]:
def classical_rate_equations(y, t, k_ET, k_TTA, k_form):
    """
    古典的速度方程式。
    
    Parameters:
    -----------
    y : array
        ポピュレーション [p_T1S0, p_S0T1, p_T1T1, p_S1S0, p_S0S1]
    t : float
        時間
    k_ET : float
        エネルギー移動速度定数
    k_TTA : float
        TTA速度定数
    k_form : float
        T1T1形成速度定数
    """
    p_T1S0, p_S0T1, p_T1T1, p_S1S0, p_S0S1 = y
    
    dp_T1S0 = -k_ET * p_T1S0 + k_ET * p_S0T1
    dp_S0T1 = k_ET * p_T1S0 - k_ET * p_S0T1
    dp_T1T1 = k_form * p_T1S0 * p_S0T1 - 2 * k_TTA * p_T1T1
    dp_S1S0 = k_TTA * p_T1T1
    dp_S0S1 = k_TTA * p_T1T1
    
    return [dp_T1S0, dp_S0T1, dp_T1T1, dp_S1S0, dp_S0S1]

# 速度定数の設定
k_ET = V_ET * 2 * np.pi  # エネルギー移動速度 (1/ps)
k_TTA = V_TTA * 2 * np.pi  # TTA速度 (1/ps)
k_form = 0.01  # T1T1形成速度 (1/ps)

# 初期条件
y0_classical = [1.0, 0.0, 0.0, 0.0, 0.0]  # [T1S0, S0T1, T1T1, S1S0, S0S1]

# ODEソルバーで解く
solution_classical = odeint(classical_rate_equations, y0_classical, tlist, 
                            args=(k_ET, k_TTA, k_form))

# 結果をプロット
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(tlist, solution_classical[:, 0], label='|T₁S₀⟩', linewidth=2.5, color='blue')
ax.plot(tlist, solution_classical[:, 1], label='|S₀T₁⟩', linewidth=2.5, color='green')
ax.plot(tlist, solution_classical[:, 2], label='|T₁T₁⟩', linewidth=2.5, color='red')
ax.plot(tlist, solution_classical[:, 3], label='|S₁S₀⟩', linewidth=2.5, color='orange')
ax.plot(tlist, solution_classical[:, 4], label='|S₀S₁⟩', linewidth=2.5, color='purple')
ax.set_xlabel('時間 (ps)')
ax.set_ylabel('ポピュレーション')
ax.set_title('古典的速度論モデルによるTTAダイナミクス')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, t_max])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.show()

print("古典的モデルの計算完了")
print(f"最終ポピュレーション:")
print(f"  |T₁S₀⟩: {solution_classical[-1, 0]:.4f}")
print(f"  |S₀T₁⟩: {solution_classical[-1, 1]:.4f}")
print(f"  |T₁T₁⟩: {solution_classical[-1, 2]:.4f}")
print(f"  |S₁S₀⟩: {solution_classical[-1, 3]:.4f}")
print(f"  |S₀S₁⟩: {solution_classical[-1, 4]:.4f}")


## 3. 量子力学的記述### 3.1 ハミルトニアン構築2分子系の全ハミルトニアンは4つの項から構成されます（修正版）：$$H = H_0 + H_{\text{ET}} + H_{\text{form}} + H_{\text{TTA}}$$#### 3.1.1 自由ハミルトニアン $H_0$各分子の自由エネルギー：$$H_0 = E_{T_1}(|T_1 S_0\rangle\langle T_1 S_0| + |S_0 T_1\rangle\langle S_0 T_1|) + 2E_{T_1}|T_1 T_1\rangle\langle T_1 T_1| + E_{S_1}(|S_1 S_0\rangle\langle S_1 S_0| + |S_0 S_1\rangle\langle S_0 S_1|)$$#### 3.1.2 エネルギー移動ハミルトニアン $H_{\text{ET}}$$$H_{\text{ET}} = V_{\text{ET}} (|T_1 S_0\rangle\langle S_0 T_1| + |S_0 T_1\rangle\langle T_1 S_0|)$$#### 3.1.3 T₁T₁形成ハミルトニアン $H_{\text{form}}$ (NEW)$$H_{\text{form}} = V_{\text{form}} (|T_1 S_0\rangle\langle T_1 T_1| + |S_0 T_1\rangle\langle T_1 T_1| + |T_1 T_1\rangle\langle T_1 S_0| + |T_1 T_1\rangle\langle S_0 T_1|)$$この項は、両分子が励起状態にある時の二体相互作用を表し、$|T_1 T_1\rangle$ 状態の形成を可能にします。この項がないと、初期状態から $|T_1 T_1\rangle$ に到達できず、TTAが起こりません。#### 3.1.4 TTA ハミルトニアン $H_{\text{TTA}}$$$H_{\text{TTA}} = V_{\text{TTA}} (|S_1 S_0\rangle\langle T_1 T_1| + |S_0 S_1\rangle\langle T_1 T_1| + |T_1 T_1\rangle\langle S_1 S_0| + |T_1 T_1\rangle\langle S_0 S_1|)$$### 3.2 時間発展シュレーディンガー方程式： $|\psi(t)\rangle = e^{-iHt/\hbar} |\psi(0)\rangle$初期状態： $|\psi(0)\rangle = |T_1 S_0\rangle$

In [ ]:
# ハミルトニアン行列の構築（簡略化された5状態系）def construct_hamiltonian_matrices():    """    5状態系のハミルトニアン行列を構築。    状態順序: |T1S0>, |S0T1>, |T1T1>, |S1S0>, |S0S1>        FIXED: H_form項を追加してTTAを可能にする    """    # 5x5ゼロ行列    H0 = np.zeros((5, 5), dtype=complex)    H_ET = np.zeros((5, 5), dtype=complex)    H_form = np.zeros((5, 5), dtype=complex)  # NEW: 形成項    H_TTA = np.zeros((5, 5), dtype=complex)        # H0: 自由ハミルトニアン (対角成分)    H0[0, 0] = E_T1          # |T1S0>    H0[1, 1] = E_T1          # |S0T1>    H0[2, 2] = 2 * E_T1      # |T1T1>    H0[3, 3] = E_S1          # |S1S0>    H0[4, 4] = E_S1          # |S0S1>        # H_ET: エネルギー移動 (|T1S0> <-> |S0T1>)    H_ET[0, 1] = V_ET    H_ET[1, 0] = V_ET        # H_form: T1T1形成 (NEW: |T1S0> <-> |T1T1>, |S0T1> <-> |T1T1>)    # これは両分子が励起状態にある時の二体相互作用を表す    # この項がないと|T1T1>状態が到達不可能でTTAが起こらない    H_form[0, 2] = V_form  # |T1S0> <-> |T1T1>    H_form[2, 0] = V_form    H_form[1, 2] = V_form  # |S0T1> <-> |T1T1>    H_form[2, 1] = V_form        # H_TTA: 三重項-三重項消滅 (|T1T1> <-> |S1S0>, |S0S1>)    H_TTA[2, 3] = V_TTA  # |T1T1> -> |S1S0>    H_TTA[3, 2] = V_TTA    H_TTA[2, 4] = V_TTA  # |T1T1> -> |S0S1>    H_TTA[4, 2] = V_TTA        # 全ハミルトニアン (FIXED: H_form追加)    H_total = H0 + H_ET + H_form + H_TTA        return H0, H_ET, H_form, H_TTA, H_total# ハミルトニアン構築H0, H_ET, H_form, H_TTA, H_total = construct_hamiltonian_matrices()print("ハミルトニアン行列構築完了（H_form項を追加）")print(f"\n全ハミルトニアン H = H0 + H_ET + H_form + H_TTA:")print(H_total)print(f"\n非ゼロ結合項:")print(f"  |T₁S₀⟩ ↔ |S₀T₁⟩: V_ET  = {V_ET:.4f} eV (エネルギー移動)")print(f"  |T₁S₀⟩ ↔ |T₁T₁⟩: V_form = {V_form:.4f} eV (形成、NEW)")print(f"  |S₀T₁⟩ ↔ |T₁T₁⟩: V_form = {V_form:.4f} eV (形成、NEW)")print(f"  |T₁T₁⟩ ↔ |S₁S₀⟩: V_TTA  = {V_TTA:.4f} eV (TTA)")print(f"  |T₁T₁⟩ ↔ |S₀S₁⟩: V_TTA  = {V_TTA:.4f} eV (TTA)")# 初期状態psi0 = np.zeros(5, dtype=complex)psi0[0] = 1.0  # |T1S0> 状態print(f"\n初期状態: |T₁S₀⟩")print(f"\nFIXED: H_form項により、|T₁S₀⟩ → |T₁T₁⟩ → |S₁S₀⟩, |S₀S₁⟩ の経路が可能になりました。")

### 3.1.4 T₁T₁形成ハミルトニアン $H_{\text{form}}$ (重要な修正)

**問題**: 元の理論では $H = H_0 + H_{\text{ET}} + H_{\text{TTA}}$ の3項のみを定義していましたが、これでは初期状態 $|T_1 S_0\rangle$ から $|T_1 T_1\rangle$ 状態に到達する経路が存在せず、TTAが決して起こりません。

**解決策**: $H_{\text{form}}$ 項を追加して $|T_1 T_1\rangle$ 状態の形成を可能にします：

$$
H_{\text{form}} = V_{\text{form}} (|T_1 S_0\rangle\langle T_1 T_1| + |S_0 T_1\rangle\langle T_1 T_1| + \text{h.c.})
$$

この項は、両分子が励起状態にある時の**二体相互作用**を表現します。物理的には以下のような相互作用から生じます：

1. **双極子-双極子相互作用**: 励起分子間の電気双極子モーメント相互作用
2. **交換相互作用**: 分子が近接した時の電子波動関数の重なり
3. **有効結合**: より高次の仮想状態を通じた実効的な結合

完全なハミルトニアンは：

$$
H = H_0 + H_{\text{ET}} + H_{\text{form}} + H_{\text{TTA}}
$$

これにより、以下の経路が可能になります：

$$
|T_1 S_0\rangle \xrightarrow{H_{\text{form}}} |T_1 T_1\rangle \xrightarrow{H_{\text{TTA}}} |S_1 S_0\rangle, |S_0 S_1\rangle
$$

In [ ]:
# 厳密な時間発展（行列の指数関数）

def exact_time_evolution(H, psi0, tlist):
    """
    厳密な時間発展を計算。
    """
    psi_t = []
    for t in tlist:
        U_t = expm(-1j * H * t)  # 時間発展演算子
        psi = U_t @ psi0
        psi_t.append(psi)
    return np.array(psi_t)

# 厳密解を計算
psi_exact = exact_time_evolution(H_total, psi0, tlist)

# ポピュレーション計算
populations_exact = np.abs(psi_exact)**2

# プロット
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(tlist, populations_exact[:, 0], label='|T₁S₀⟩', linewidth=2.5, color='blue')
ax.plot(tlist, populations_exact[:, 1], label='|S₀T₁⟩', linewidth=2.5, color='green')
ax.plot(tlist, populations_exact[:, 2], label='|T₁T₁⟩', linewidth=2.5, color='red')
ax.plot(tlist, populations_exact[:, 3], label='|S₁S₀⟩', linewidth=2.5, color='orange')
ax.plot(tlist, populations_exact[:, 4], label='|S₀S₁⟩', linewidth=2.5, color='purple')
ax.set_xlabel('時間 (ps)')
ax.set_ylabel('ポピュレーション')
ax.set_title('量子力学的記述（厳密解）')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, t_max])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.show()

print("量子力学的厳密解の計算完了")
print(f"最終ポピュレーション:")
for i, state in enumerate(['|T₁S₀⟩', '|S₀T₁⟩', '|T₁T₁⟩', '|S₁S₀⟩', '|S₀S₁⟩']):
    print(f"  {state}: {populations_exact[-1, i]:.4f}")


### 3.3 鈴木-トロッター分解

ハミルトニアン $H = H_0 + H_{\text{ET}} + H_{\text{TTA}}$ の時間発展を近似します。

**2次トロッター分解（Strang splitting）**:

$$e^{-iH\Delta t} \approx e^{-iH_0\Delta t/2} e^{-iH_{\text{ET}}\Delta t/2} e^{-iH_{\text{TTA}}\Delta t} e^{-iH_{\text{ET}}\Delta t/2} e^{-iH_0\Delta t/2}$$

全時間 $t$ に対して、ステップ数 $N = t/\Delta t$ の繰り返し適用：

$$e^{-iHt} \approx \left(e^{-iH_0\Delta t/2} e^{-iH_{\text{ET}}\Delta t/2} e^{-iH_{\text{TTA}}\Delta t} e^{-iH_{\text{ET}}\Delta t/2} e^{-iH_0\Delta t/2}\right)^N$$


In [ ]:
def trotter_time_evolution(H_terms, psi0, t_final, dt, order=2):    """    鈴木-トロッター分解による時間発展。        Parameters:    -----------    H_terms : list of ndarray        ハミルトニアン項のリスト [H0, H_ET, H_form, H_TTA]    psi0 : ndarray        初期状態    t_final : float        最終時間    dt : float        時間ステップ    order : int        トロッター分解の次数    """    n_steps = int(t_final / dt)    psi = psi0.copy()        if order == 1:        # 1次トロッター (Lie-Trotter)        for _ in range(n_steps):            for H in H_terms:                psi = expm(-1j * H * dt) @ psi    elif order == 2:        # 2次トロッター (Strang splitting)        for _ in range(n_steps):            # 前半            for H in H_terms:                psi = expm(-1j * H * dt / 2) @ psi            # 後半（逆順）            for H in reversed(H_terms):                psi = expm(-1j * H * dt / 2) @ psi        return psi# トロッター分解による時間発展 (FIXED: H_form追加)H_terms = [H0, H_ET, H_form, H_TTA]psi_trotter_list = []for t in tlist:    psi_t = trotter_time_evolution(H_terms, psi0, t, dt_trotter, order=trotter_order)    psi_trotter_list.append(psi_t)psi_trotter = np.array(psi_trotter_list)populations_trotter = np.abs(psi_trotter)**2# トロッター分解と厳密解の比較fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))# 厳密解for i, (label, color) in enumerate(zip(    ['|T₁S₀⟩', '|S₀T₁⟩', '|T₁T₁⟩', '|S₁S₀⟩', '|S₀S₁⟩'],    ['blue', 'green', 'red', 'orange', 'purple'])):    ax1.plot(tlist, populations_exact[:, i], label=label, linewidth=2.5, color=color)ax1.set_xlabel('時間 (ps)')ax1.set_ylabel('ポピュレーション')ax1.set_title('厳密解')ax1.legend(loc='best')ax1.grid(True, alpha=0.3)ax1.set_xlim([0, t_max])ax1.set_ylim([0, 1.05])# トロッター近似for i, (label, color) in enumerate(zip(    ['|T₁S₀⟩', '|S₀T₁⟩', '|T₁T₁⟩', '|S₁S₀⟩', '|S₀S₁⟩'],    ['blue', 'green', 'red', 'orange', 'purple'])):    ax2.plot(tlist, populations_trotter[:, i], label=label, linewidth=2.5, color=color, linestyle='--')ax2.set_xlabel('時間 (ps)')ax2.set_ylabel('ポピュレーション')ax2.set_title(f'トロッター分解 (order={trotter_order}, dt={dt_trotter} ps)')ax2.legend(loc='best')ax2.grid(True, alpha=0.3)ax2.set_xlim([0, t_max])ax2.set_ylim([0, 1.05])plt.tight_layout()plt.show()# 誤差計算error = np.max(np.abs(populations_exact - populations_trotter))print(f"\nトロッター分解の精度:")print(f"  最大誤差: {error:.6f}")print(f"  次数: {trotter_order}")print(f"  ステップサイズ: {dt_trotter} ps")

## 4. Qubit表現によるシミュレーション（Qiskit）

### 4.1 状態の符号化

3準位系（S₀, T₁, S₁）を2量子ビットで符号化します：

$$
\begin{aligned}
|S_0\rangle &\rightarrow |00\rangle \\
|T_1\rangle &\rightarrow |01\rangle \\
|S_1\rangle &\rightarrow |10\rangle \\
&\quad |11\rangle \text{ (未使用)}
\end{aligned}
$$

### 4.2 2分子系の符号化

2分子系は4量子ビットで表現されます（各分子に2量子ビット）：

$$
\begin{aligned}
|T_1 S_0\rangle &\rightarrow |0100\rangle \\
|S_0 T_1\rangle &\rightarrow |0001\rangle \\
|T_1 T_1\rangle &\rightarrow |0101\rangle \\
|S_1 S_0\rangle &\rightarrow |1000\rangle \\
|S_0 S_1\rangle &\rightarrow |0010\rangle
\end{aligned}
$$

### 4.3 量子回路の構成

1. **初期化**: $|0100\rangle$ 状態の準備
2. **トロッター分解**: 各ハミルトニアン項を量子ゲートに分解
3. **測定**: 全量子ビットを計算基底で測定


In [ ]:
if QISKIT_AVAILABLE:    print("=== Qubit表現シミュレーション（Qiskit） ===\n")        # 状態インデックスのマッピング    # 5状態を4量子ビット（16次元）にマッピング    state_to_qubit = {        0: 0b0100,  # |T1S0> -> |0100>        1: 0b0001,  # |S0T1> -> |0001>        2: 0b0101,  # |T1T1> -> |0101>        3: 0b1000,  # |S1S0> -> |1000>        4: 0b0010   # |S0S1> -> |0010>    }        # 16次元の初期状態ベクトル    psi0_qubit = np.zeros(16, dtype=complex)    psi0_qubit[state_to_qubit[0]] = 1.0  # |T1S0> = |0100>        print(f"初期状態 (qubit表現): |0100⟩ (10進数: {state_to_qubit[0]})")    print(f"状態空間次元: 16 (4 qubits)")        # 16x16のハミルトニアン行列を構築    H0_qubit = np.zeros((16, 16), dtype=complex)    H_ET_qubit = np.zeros((16, 16), dtype=complex)    H_form_qubit = np.zeros((16, 16), dtype=complex)  # NEW: 形成項    H_TTA_qubit = np.zeros((16, 16), dtype=complex)        # 対角成分（H0）    H0_qubit[state_to_qubit[0], state_to_qubit[0]] = E_T1    H0_qubit[state_to_qubit[1], state_to_qubit[1]] = E_T1    H0_qubit[state_to_qubit[2], state_to_qubit[2]] = 2 * E_T1    H0_qubit[state_to_qubit[3], state_to_qubit[3]] = E_S1    H0_qubit[state_to_qubit[4], state_to_qubit[4]] = E_S1        # エネルギー移動（H_ET）    H_ET_qubit[state_to_qubit[0], state_to_qubit[1]] = V_ET    H_ET_qubit[state_to_qubit[1], state_to_qubit[0]] = V_ET        # T1T1形成（H_form）NEW    H_form_qubit[state_to_qubit[0], state_to_qubit[2]] = V_form    H_form_qubit[state_to_qubit[2], state_to_qubit[0]] = V_form    H_form_qubit[state_to_qubit[1], state_to_qubit[2]] = V_form    H_form_qubit[state_to_qubit[2], state_to_qubit[1]] = V_form        # TTA（H_TTA）    H_TTA_qubit[state_to_qubit[2], state_to_qubit[3]] = V_TTA    H_TTA_qubit[state_to_qubit[3], state_to_qubit[2]] = V_TTA    H_TTA_qubit[state_to_qubit[2], state_to_qubit[4]] = V_TTA    H_TTA_qubit[state_to_qubit[4], state_to_qubit[2]] = V_TTA        H_total_qubit = H0_qubit + H_ET_qubit + H_form_qubit + H_TTA_qubit  # FIXED        print("\n4量子ビットハミルトニアン構築完了")    else:    print("Qiskitが利用できないため、qubit表現のシミュレーションをスキップします。")

In [ ]:
if QISKIT_AVAILABLE:    # Statevectorシミュレーション    print("\n### Statevectorシミュレーション ###\n")        # トロッター分解による時間発展 (FIXED: H_form_qubit追加)    H_terms_qubit = [H0_qubit, H_ET_qubit, H_form_qubit, H_TTA_qubit]    psi_qubit_list = []        for t in tlist:        psi_t = trotter_time_evolution(H_terms_qubit, psi0_qubit, t, dt_trotter, order=trotter_order)        psi_qubit_list.append(psi_t)        psi_qubit = np.array(psi_qubit_list)        # 物理的な状態のポピュレーションを抽出    populations_qubit = np.zeros((len(tlist), 5))    for i in range(5):        populations_qubit[:, i] = np.abs(psi_qubit[:, state_to_qubit[i]])**2        # プロット    fig, ax = plt.subplots(figsize=(12, 6))    for i, (label, color) in enumerate(zip(        ['|T₁S₀⟩', '|S₀T₁⟩', '|T₁T₁⟩', '|S₁S₀⟩', '|S₀S₁⟩'],        ['blue', 'green', 'red', 'orange', 'purple'])):        ax.plot(tlist, populations_qubit[:, i], label=label, linewidth=2.5, color=color)        ax.set_xlabel('時間 (ps)')    ax.set_ylabel('ポピュレーション')    ax.set_title('Qubit表現 - Statevectorシミュレーション（Qiskit）')    ax.legend(loc='best')    ax.grid(True, alpha=0.3)    ax.set_xlim([0, t_max])    ax.set_ylim([0, 1.05])    plt.tight_layout()    plt.show()        print("Statevectorシミュレーション完了")    print(f"\n最終ポピュレーション:")    for i, state in enumerate(['|T₁S₀⟩', '|S₀T₁⟩', '|T₁T₁⟩', '|S₁S₀⟩', '|S₀S₁⟩']):        print(f"  {state}: {populations_qubit[-1, i]:.4f}")

In [ ]:
if QISKIT_AVAILABLE:
    # ショットシミュレーション
    print("\n### ショットシミュレーション ###\n")
    
    # 最終状態でのショットシミュレーション
    psi_final = psi_qubit[-1]
    
    # 確率分布から測定結果をサンプリング
    probabilities = np.abs(psi_final)**2
    probabilities = probabilities / np.sum(probabilities)  # 正規化
    
    # ショット数分サンプリング
    measurement_results = np.random.choice(16, size=n_shots, p=probabilities)
    
    # 測定結果をカウント
    counts = {i: 0 for i in range(16)}
    for result in measurement_results:
        counts[result] += 1
    
    # 物理的な状態のカウントを抽出
    state_counts = {}
    state_names = ['|T₁S₀⟩', '|S₀T₁⟩', '|T₁T₁⟩', '|S₁S₀⟩', '|S₀S₁⟩']
    for i, name in enumerate(state_names):
        state_counts[name] = counts[state_to_qubit[i]]
    
    # プロット
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(state_counts.keys(), [c/n_shots for c in state_counts.values()], 
           color=['blue', 'green', 'red', 'orange', 'purple'], alpha=0.7)
    ax.set_ylabel('測定確率')
    ax.set_title(f'Qubit表現 - ショットシミュレーション（{n_shots} shots）')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 1.0])
    plt.tight_layout()
    plt.show()
    
    print(f"ショットシミュレーション完了 ({n_shots} shots)")
    print(f"\n測定結果:")
    for name, count in state_counts.items():
        print(f"  {name}: {count/n_shots:.4f} ({count} counts)")


### 4.4 量子回路の解析

量子回路のリソース分析を行います：

- **量子ビット数**: 使用する量子ビットの総数
- **ゲート数**: 量子回路に含まれる量子ゲートの総数
- **回路深さ**: 量子回路の最長パスの長さ（並列実行を考慮）


In [ ]:
if QISKIT_AVAILABLE:
    print("\n### 量子回路解析 ###\n")
    
    # 簡単な量子回路の構築（デモ用）
    qr = QuantumRegister(4, 'q')
    cr = ClassicalRegister(4, 'c')
    qc = QuantumCircuit(qr, cr)
    
    # 初期状態の準備: |0100>
    qc.x(qr[1])  # 量子ビット1をフリップ
    qc.barrier()
    
    # トロッターステップ（簡略版）
    for _ in range(5):  # 5ステップのデモ
        # H0の近似（RZゲート）
        qc.rz(E_T1 * dt_trotter, qr[0])
        qc.rz(E_T1 * dt_trotter, qr[2])
        
        # エネルギー移動（CNOTとRYゲート）
        qc.cx(qr[1], qr[3])
        qc.ry(V_ET * dt_trotter, qr[1])
        qc.cx(qr[1], qr[3])
        
        qc.barrier()
    
    # 測定
    qc.measure(qr, cr)
    
    # 回路統計
    num_qubits = qc.num_qubits
    num_gates = len(qc.data)
    depth = qc.depth()
    
    print(f"量子回路統計:")
    print(f"  量子ビット数: {num_qubits}")
    print(f"  ゲート数: {num_gates}")
    print(f"  回路深さ: {depth}")
    print(f"  トロッターステップ数: 5 (デモ)")
    
    # 回路の可視化
    print("\n量子回路:")
    print(qc.draw(output='text', fold=-1))
    
else:
    print("Qiskitが利用できないため、回路解析をスキップします。")


## 5. Qudit表現によるシミュレーション（MQT）

### 5.1 Qutrit（3準位量子系）

Quditは $d$ 準位量子系です。3準位の場合、qutritと呼ばれます。

### 5.2 状態の直接表現

3準位系を直接3準位quditで表現します：

$$
\begin{aligned}
|S_0\rangle &\rightarrow |0\rangle_3 \\
|T_1\rangle &\rightarrow |1\rangle_3 \\
|S_1\rangle &\rightarrow |2\rangle_3
\end{aligned}
$$

### 5.3 2分子系の表現

2分子系は2つのqutritで表現されます：

$$|X_A X_B\rangle \rightarrow |i\rangle_3 \otimes |j\rangle_3$$

全状態空間の次元は $3 \times 3 = 9$ です。

状態のマッピング：
- $|T_1 S_0\rangle \rightarrow |10\rangle_3$
- $|S_0 T_1\rangle \rightarrow |01\rangle_3$
- $|T_1 T_1\rangle \rightarrow |11\rangle_3$
- $|S_1 S_0\rangle \rightarrow |20\rangle_3$
- $|S_0 S_1\rangle \rightarrow |02\rangle_3$

### 5.4 利点

- **効率的**: Qubitの4量子ビット（16次元）に対し、Quditは2 qutrit（9次元）
- **自然な表現**: 物理的な3準位系を直接表現
- **少ないゲート数**: より少ない量子演算で実装可能


In [ ]:
if MQT_AVAILABLE:    print("=== Qudit表現シミュレーション（MQT） ===\n")        # Quditハミルトニアン（そのまま使用可能）    # 状態順序: 0=T1S0, 1=S0T1, 2=T1T1, 3=S1S0, 4=S0S1        print(f"状態空間次元: 9 (2 qutrits, but using 5 physical states)")    print("Qudit表現では、物理的な5状態を直接扱います。")        # MQTシミュレータの初期化    try:        simulator = MQTSimulator(num_qudits=2, qudit_dim=3)        print("\nMQTシミュレータ初期化成功")                # ハミルトニアン項のリスト (FIXED: H_form追加)        H_terms_qudit = [H0, H_ET, H_form, H_TTA]                # 初期状態（5次元）        psi0_qudit = np.zeros(5, dtype=complex)        psi0_qudit[0] = 1.0  # |T1S0>                print(f"初期状態: |T₁S₀⟩")            except Exception as e:        print(f"MQTシミュレータ初期化エラー: {e}")        print("簡略化されたシミュレーションを実行します。")        else:    print("MQTモジュールが利用できないため、qudit表現のシミュレーションをスキップします。")

In [ ]:
if MQT_AVAILABLE:
    # Statevectorシミュレーション（簡略版）
    print("\n### Statevectorシミュレーション ###\n")
    
    # トロッター分解を使用（qubitと同じアルゴリズム）
    psi_qudit_list = []
    
    for t in tlist:
        psi_t = trotter_time_evolution(H_terms_qudit, psi0_qudit, t, dt_trotter, order=trotter_order)
        psi_qudit_list.append(psi_t)
    
    psi_qudit = np.array(psi_qudit_list)
    populations_qudit = np.abs(psi_qudit)**2
    
    # プロット
    fig, ax = plt.subplots(figsize=(12, 6))
    for i, (label, color) in enumerate(zip(
        ['|T₁S₀⟩', '|S₀T₁⟩', '|T₁T₁⟩', '|S₁S₀⟩', '|S₀S₁⟩'],
        ['blue', 'green', 'red', 'orange', 'purple'])):
        ax.plot(tlist, populations_qudit[:, i], label=label, linewidth=2.5, color=color)
    
    ax.set_xlabel('時間 (ps)')
    ax.set_ylabel('ポピュレーション')
    ax.set_title('Qudit表現 - Statevectorシミュレーション（MQT）')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, t_max])
    ax.set_ylim([0, 1.05])
    plt.tight_layout()
    plt.show()
    
    print("Statevectorシミュレーション完了")
    print(f"\n最終ポピュレーション:")
    for i, state in enumerate(['|T₁S₀⟩', '|S₀T₁⟩', '|T₁T₁⟩', '|S₁S₀⟩', '|S₀S₁⟩']):
        print(f"  {state}: {populations_qudit[-1, i]:.4f}")


In [ ]:
if MQT_AVAILABLE:
    # ショットシミュレーション
    print("\n### ショットシミュレーション ###\n")
    
    # 最終状態でのショットシミュレーション
    psi_final_qudit = psi_qudit[-1]
    
    # 確率分布から測定結果をサンプリング
    probabilities_qudit = np.abs(psi_final_qudit)**2
    probabilities_qudit = probabilities_qudit / np.sum(probabilities_qudit)
    
    # ショット数分サンプリング
    measurement_results_qudit = np.random.choice(5, size=n_shots, p=probabilities_qudit)
    
    # 測定結果をカウント
    counts_qudit = np.bincount(measurement_results_qudit, minlength=5)
    
    # プロット
    fig, ax = plt.subplots(figsize=(10, 6))
    state_names = ['|T₁S₀⟩', '|S₀T₁⟩', '|T₁T₁⟩', '|S₁S₀⟩', '|S₀S₁⟩']
    ax.bar(state_names, counts_qudit/n_shots, 
           color=['blue', 'green', 'red', 'orange', 'purple'], alpha=0.7)
    ax.set_ylabel('測定確率')
    ax.set_title(f'Qudit表現 - ショットシミュレーション（{n_shots} shots）')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 1.0])
    plt.tight_layout()
    plt.show()
    
    print(f"ショットシミュレーション完了 ({n_shots} shots)")
    print(f"\n測定結果:")
    for i, name in enumerate(state_names):
        print(f"  {name}: {counts_qudit[i]/n_shots:.4f} ({counts_qudit[i]} counts)")


### 5.5 量子回路の解析

Qudit量子回路のリソース分析：

- **Qudit数**: 2 qutrits
- **状態空間次元**: 9（実際に使用するのは5状態）
- **ゲート数**: Qubit表現より少ない
- **回路深さ**: Qubit表現より浅い可能性


In [ ]:
if MQT_AVAILABLE:
    print("\n### 量子回路解析 ###\n")
    
    # Qudit回路の統計（理論値）
    num_qutrits = 2
    state_space_dim = 9
    actual_states = 5
    
    # トロッター分解における演算数の推定
    # 各トロッターステップで3つのハミルトニアン項を適用
    operations_per_step = 3  # H0, H_ET, H_TTA
    total_operations = operations_per_step * n_trotter_steps
    
    print(f"Qudit量子回路統計:")
    print(f"  Qudit数: {num_qutrits} (qutrits)")
    print(f"  Qudit次元: 3")
    print(f"  状態空間次元: {state_space_dim}")
    print(f"  使用する物理的状態数: {actual_states}")
    print(f"  トロッターステップ数: {n_trotter_steps}")
    print(f"  推定演算数: {total_operations}")
    print(f"  推定回路深さ: {n_trotter_steps * operations_per_step}")
    
    # Qubit vs Qudit の比較
    print(f"\n=== Qubit vs Qudit 比較 ===")
    print(f"  量子資源:")
    print(f"    Qubit: 4 qubits (状態空間: 16次元)")
    print(f"    Qudit: 2 qutrits (状態空間: 9次元)")
    print(f"  効率:")
    print(f"    Quditの方が約 {16/9:.2f}x コンパクト")
    
else:
    print("MQTモジュールが利用できないため、回路解析をスキップします。")


## 6. 結果の比較と考察

### 6.1 全シミュレーション結果の比較

古典的モデル、量子力学的厳密解、Qubit表現、Qudit表現の結果を比較します。


In [ ]:
# 全結果の比較プロット
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 古典的モデル
ax = axes[0, 0]
for i, (label, color) in enumerate(zip(
    ['|T₁S₀⟩', '|S₀T₁⟩', '|T₁T₁⟩', '|S₁S₀⟩', '|S₀S₁⟩'],
    ['blue', 'green', 'red', 'orange', 'purple'])):
    ax.plot(tlist, solution_classical[:, i], label=label, linewidth=2, color=color)
ax.set_xlabel('時間 (ps)')
ax.set_ylabel('ポピュレーション')
ax.set_title('古典的速度論モデル')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, t_max])
ax.set_ylim([0, 1.05])

# 量子力学的厳密解
ax = axes[0, 1]
for i, (label, color) in enumerate(zip(
    ['|T₁S₀⟩', '|S₀T₁⟩', '|T₁T₁⟩', '|S₁S₀⟩', '|S₀S₁⟩'],
    ['blue', 'green', 'red', 'orange', 'purple'])):
    ax.plot(tlist, populations_exact[:, i], label=label, linewidth=2, color=color)
ax.set_xlabel('時間 (ps)')
ax.set_ylabel('ポピュレーション')
ax.set_title('量子力学的厳密解')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, t_max])
ax.set_ylim([0, 1.05])

# Qubit表現
ax = axes[1, 0]
if QISKIT_AVAILABLE:
    for i, (label, color) in enumerate(zip(
        ['|T₁S₀⟩', '|S₀T₁⟩', '|T₁T₁⟩', '|S₁S₀⟩', '|S₀S₁⟩'],
        ['blue', 'green', 'red', 'orange', 'purple'])):
        ax.plot(tlist, populations_qubit[:, i], label=label, linewidth=2, color=color)
    ax.set_title('Qubit表現 (Qiskit)')
else:
    ax.text(0.5, 0.5, 'Qiskit not available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Qubit表現 (Qiskit) - 利用不可')
ax.set_xlabel('時間 (ps)')
ax.set_ylabel('ポピュレーション')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, t_max])
ax.set_ylim([0, 1.05])

# Qudit表現
ax = axes[1, 1]
if MQT_AVAILABLE:
    for i, (label, color) in enumerate(zip(
        ['|T₁S₀⟩', '|S₀T₁⟩', '|T₁T₁⟩', '|S₁S₀⟩', '|S₀S₁⟩'],
        ['blue', 'green', 'red', 'orange', 'purple'])):
        ax.plot(tlist, populations_qudit[:, i], label=label, linewidth=2, color=color)
    ax.set_title('Qudit表現 (MQT)')
else:
    ax.text(0.5, 0.5, 'MQT not available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Qudit表現 (MQT) - 利用不可')
ax.set_xlabel('時間 (ps)')
ax.set_ylabel('ポピュレーション')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, t_max])
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

print("\n全シミュレーション結果の比較完了")


### 6.2 考察

1. **古典モデル vs 量子モデル**:
   - 古典的速度論モデルは近似的な記述
   - 量子力学的モデルは波動関数の干渉効果を含む
   - 短時間では両者に顕著な違いが現れる可能性

2. **Qubit vs Qudit表現**:
   - **Qubit表現**: 4量子ビット（16次元空間）
   - **Qudit表現**: 2 qutrit（9次元空間）
   - Qudit表現の方がよりコンパクトで効率的

3. **鈴木-トロッター分解の精度**:
   - ステップサイズ $\Delta t$ が小さいほど精度向上
   - 高次の分解（order=4）はより正確だが計算コスト増加
   - 本シミュレーションでは2次分解で十分な精度

4. **量子回路のリソース**:
   - Qubitゲート数: 多い（パウリゲートの組み合わせ）
   - Quditゲート数: 少ない（直接3準位演算）
   - 実際の量子ハードウェアでは、Qubitが現状主流

5. **物理的意義**:
   - エネルギー移動により初期状態 $|T_1 S_0\rangle$ から $|S_0 T_1\rangle$ へ遷移
   - 両分子が励起状態になると、TTA過程で一重項励起状態が生成
   - TTA過程はアップコンバージョン光学素子の基礎原理


## 7. まとめ

本チュートリアルでは、三重項-三重項消滅(TTA)過程の量子ダイナミクスを、古典的、qubit、qudit表現で記述し、シミュレーションしました。

### 7.1 実装内容

1. **古典的速度論モデル**: 速度方程式による記述
2. **量子力学的記述**: ハミルトニアンと厳密な時間発展
3. **鈴木-トロッター分解**: 時間発展の数値計算アルゴリズム
4. **Qubit表現（Qiskit）**: 4量子ビットによる実装
   - Statevectorシミュレーション
   - ショットシミュレーション
   - 回路解析（qubit数、ゲート数、深さ）
5. **Qudit表現（MQT）**: 2 qutritによる直接実装
   - Statevectorシミュレーション
   - ショットシミュレーション
   - 回路解析（qudit数、ゲート数、深さ）

### 7.2 主な成果

- **多角的アプローチ**: 同一の物理過程を複数の方法で記述・シミュレーション
- **精度検証**: トロッター分解の精度を厳密解と比較
- **効率性評価**: QubitとQudit表現のリソース比較
- **可視化**: 各手法の結果を統一的にプロット

### 7.3 今後の展開

1. **より複雑な系**: 3分子以上の系への拡張
2. **環境効果**: デコヒーレンスや散逸過程の導入
3. **実験パラメータ**: 実際の色素分子のパラメータでのシミュレーション
4. **実機実行**: 実際の量子コンピュータでの実行と検証

### 7.4 参考文献

詳細な理論と数式は [`triplet_triplet_annihilation_theory.md`](triplet_triplet_annihilation_theory.md) を参照してください。

---

**作成日**: 2025-10-13  
**バージョン**: 1.0  
**環境**: Python 3.9+, QuTiP, Qiskit, MQT
